<a href="https://colab.research.google.com/github/AnastasiyaPunko/NIPT/blob/main/NIPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
conda activate ngs

# QC

In [ ]:
fastqc -t 4 -o /home/punko_a/nipt_140526/qc/ *.fq.gz

# Alignment

In [ ]:
bwa mem -R '@RG\tID:GSS5-0798\tSM:NIPT\tPL:ILLUMINA' -t 3 /home/punko_a/genom/Homo_sapiens_assembly37.fasta /data/NIPT_281026/11_L000_R1.fq.gz > 11.sam

In [ ]:
samtools view -Sb 1-1.sam > 1-1.bam

In [ ]:
samtools sort -m 3G -o 1-1_sort.bam 1-1.bam

In [ ]:
samtools index 1-1_sort.bam

In [ ]:
samtools view -c .bam

idxstats
flagstat
quickcheck -v

# MAPQ

In [ ]:
# mapped %; primary mapped %
samtools flagstat *_sort.fixed.bam

In [ ]:
# Количество primary mapped reads с MAPQ >= 30:
samtools view -c -F 0x904 -q 30 *_sort.fixed.bam

# Количество всех primary mapped reads (MAPQ без порога):
samtools view -c -F 0x904 *_sort.fixed.bam

In [ ]:
# Проверка распределения длины SE reads по BAM:
samtools view -F 0x900 sample.bam | awk '{print length($10)}' | sort | uniq -c

In [ ]:
# reads by chr
samtools idxstats *_sort.fixed.bam

In [ ]:
#new bam with MAPQ >30
samtools view -b -q 30 -F 0x904 \
  -o NIPT_1_MAPQ30.bam \
  1_sort.fixed.bam

samtools index NIPT_11_MAPQ30.bam

# SeqFF

*Kim S.K., et al. Determination of fetal DNA fraction from the plasma of pregnant women using sequence read counts. Prenat. Diagn. 2015;35:810–815. doi: 10.1002/pd.4615.*


In [ ]:
conda activate /home/punko_a/conda_envs/seqff

In [ ]:
export R_LIBS_USER="$CONDA_PREFIX/lib/R/library"
which Rscript
Rscript --version

In [ ]:
Rscript seqff.r \
  -f /data/nipt_140526/align/NIPT_1-1_MAPQ30.bam \
  -o /data/nipt_140526/seqff/NIPT_1-1_seqff.csv

# Fetal sex determination

In [ ]:
conda activate ngs

In [ ]:
echo -e "bam\tautosomal_q30\tY_Yq11_q30\tY_per_million" > sex_y.tsv
for b in /data/NIPT_281026/align/*_MAPQ30.bam; do
  auto=$(samtools idxstats $b | awk '$1 ~ /^([1-9]|1[0-9]|2[0-2])$/ {s+=$3} END{print s}')
  y=$(samtools view -c -F 0x904 -q 30 $b Y:14000000-28000000)
  echo -e "$(basename $b)\t$auto\t$y\t$(awk -v y=$y -v a=$auto 'BEGIN{printf "%.2f", y*1e6/a}')" >> sex_y.tsv
done
column -t sex_y.tsv

# Z score

samples.tsv

In [ ]:
# TAB-separated. role: ref | test. group: replicates of ONE sample share a group. expected: T21|T18|T13|euploid (test only).
# Edit the bam paths to your real file names (MAPQ30 BAMs with .bai). "-" = use already counted <outdir>/counts/<id>.npy
sample_id	bam	role	group	expected
1-1	/data/nipt_140526/align/NIPT_1-1_MAPQ30.bam	ref	1
1-2	/data/nipt_140526/align/NIPT_1-2_MAPQ30.bam	ref	1
1-3	/data/nipt_140526/align/NIPT_1-3_MAPQ30.bam	ref	1
2-1	/data/nipt_140526/align/NIPT_2-1_MAPQ30.bam	ref	2
2-2	/data/nipt_140526/align/NIPT_2-2_MAPQ30.bam	ref	2
2-3	/data/nipt_140526/align/NIPT_2-3_MAPQ30.bam	ref	2
DNA-1	/data/NIPT_281026/align/NIPT_DNA-1_MAPQ30.bam	ref	DNA-1
NC-1	/data/nipt_140526/align/NIPT_NC-1_MAPQ30.bam	ref	NC
NC-2	/data/nipt_140526/align/NIPT_NC-2_MAPQ30.bam	ref	NC
NC-3	/data/nipt_140526/align/NIPT_NC-3_MAPQ30.bam	ref	NC
6	/data/NIPT_281026/align/NIPT_6_MAPQ30.bam	test	6
7	/data/NIPT_281026/align/NIPT_7_MAPQ30.bam	test	7
8	/data/NIPT_281026/align/NIPT_8_MAPQ30.bam	test	8
9	/data/NIPT_281026/align/NIPT_9_MAPQ30.bam	test	9
10	/data/NIPT_281026/align/NIPT_10_MAPQ30.bam	test	10
11	/data/NIPT_281026/align/5/NIPT_11_MAPQ30.bam	test	6
12	/data/NIPT_281026/align/5/NIPT_12_MAPQ30.bam	test	7
13	/data/NIPT_281026/align/5/NIPT_13_MAPQ30.bam	test	8
14	/data/NIPT_281026/align/5/NIPT_14_MAPQ30.bam	test	9
15	/data/NIPT_281026/align/5/NIPT_15_MAPQ30.bam	test	10

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
nipt_zscore.py - chromosome Z-scores (21/18/13 + all autosomes + X) from BAM files and a 100-kb BED with GC / N-ratio.

usage:
  python3 nipt_zscore.py --bed hs37d5_100kb_gc_noPAR_new.bed --samples samples.tsv --outdir z_out [options]

samples.tsv (TAB-separated, header required):
  sample_id  bam  role  [group]  [expected]
  role     = ref | test
  group    = optional; replicates of one sample share a group (1-1,1-2,1-3 -> "1")
  expected = optional for test samples: T21 | T18 | T13 | euploid  -> PASS/FAIL verdict
  bam      = path to BAM; "-" if a counts file already exists in <outdir>/counts/<sample_id>.npy

Algorithm
  1. reads (primary, mapped, MAPQ >= --mapq) are counted per BED window by the leftmost position
  2. windows removed: N-ratio > --max-nratio, GC undefined; then, from the REFERENCE samples only, windows with low/high
     median coverage or unstable coverage (mappability of 48-nt reads)
  3. per sample: GC correction (smooth curve count/expected vs GC from the BED)
  4. chromosome fraction = corrected reads of the chromosome / corrected reads of all kept autosomal windows
  5. Z = (fraction - mean_ref) / SD_ref   (reference samples only)
  Reference sanity: leave-one-out Z of every reference sample (an aneuploid or odd reference shows up here).
"""
import argparse
import csv
import os
import shutil
import struct
import subprocess
import sys
import zlib
from array import array

import numpy as np

AUTO = [str(i) for i in range(1, 23)]
TARGET = ["21", "18", "13"]


# ----------------------------------------------------------------------------- BED
def load_bed(path):
    ch, st, en, gc, nr = [], [], [], [], []
    with open(path) as f:
        for line in f:
            if not line.strip() or line.startswith(("#", "track", "browser")):
                continue
            p = line.rstrip("\n").split("\t")
            c = p[0][3:] if p[0].lower().startswith("chr") else p[0]
            ch.append(c)
            st.append(int(p[1]))
            en.append(int(p[2]))
            gc.append(float(p[3]) if p[3] not in ("", "NA", "nan", ".") else np.nan)
            nr.append(float(p[4]))
    bed = {"chr": np.array(ch), "start": np.array(st), "end": np.array(en), "gc": np.array(gc), "nr": np.array(nr)}
    idx = {}
    for c in dict.fromkeys(ch):
        rows = np.nonzero(bed["chr"] == c)[0]
        if not (np.diff(rows) == 1).all():
            sys.exit("BED must be sorted with every chromosome in one block")
        idx[c] = (rows[0], bed["start"][rows], bed["end"][rows])
    bed["idx"] = idx
    return bed


# ----------------------------------------------------------------------------- counting
def _accumulate(chrom_arr, pos, bed, counts):
    for ch in np.unique(chrom_arr):
        name = ch.decode() if isinstance(ch, bytes) else str(ch)
        name = name[3:] if name.lower().startswith("chr") else name
        if name not in bed["idx"]:
            continue
        r0, st, en = bed["idx"][name]
        p = pos[chrom_arr == ch] - 1
        i = np.searchsorted(st, p, side="right") - 1
        ok = (i >= 0) & (p < en[np.clip(i, 0, None)])
        counts += np.bincount(r0 + i[ok], minlength=len(counts))


def _bgzf_blocks(path):
    with open(path, "rb") as f:
        while True:
            h = f.read(18)
            if len(h) < 18:
                return
            xlen = struct.unpack("<H", h[10:12])[0]
            extra = h[12:18] + (f.read(xlen - 6) if xlen > 6 else b"")
            bsize = struct.unpack("<H", extra[4:6])[0]
            cdata = f.read(bsize - xlen - 19)
            f.read(8)
            if cdata:
                yield zlib.decompress(cdata, -15)


def _count_pure_python(bam, bed, mapq):
    """Fallback when samtools is not in PATH (slow, ~2 min per 10 M reads)."""
    cols = {k: array(t) for k, t in dict(rid="h", pos="i", mapq="B", flag="H").items()}
    refs, buf, started = [], bytearray(), False

    def proc(p):
        L, un = len(buf), struct.unpack_from
        while p + 4 <= L:
            bs = un("<i", buf, p)[0]
            if p + 4 + bs > L:
                break
            rid, pos, _lrn, mq, _b, _nc, fl, _ls = un("<iiBBHHHi", buf, p + 4)
            cols["rid"].append(rid)
            cols["pos"].append(pos)
            cols["mapq"].append(mq)
            cols["flag"].append(fl)
            p += 4 + bs
        return p

    for blk in _bgzf_blocks(bam):
        buf += blk
        if not started:
            lt = struct.unpack_from("<i", buf, 4)[0]
            if len(buf) < 12 + lt:
                continue
            nref = struct.unpack_from("<i", buf, 8 + lt)[0]
            p, ok, refs = 12 + lt, True, []
            for _ in range(nref):
                if p + 4 > len(buf):
                    ok = False
                    break
                l = struct.unpack_from("<i", buf, p)[0]
                if p + 8 + l > len(buf):
                    ok = False
                    break
                refs.append(bytes(buf[p + 4:p + 3 + l]).decode())
                p += 8 + l
            if not ok:
                continue
            del buf[:p]
            started = True
        if len(buf) > (1 << 20):
            del buf[:proc(0)]
    del buf[:proc(0)]
    rid, pos = np.frombuffer(cols["rid"], "h"), np.frombuffer(cols["pos"], "i").astype(np.int64)
    mq, fl = np.frombuffer(cols["mapq"], "B"), np.frombuffer(cols["flag"], "H")
    sel = ((fl & 0x904) == 0) & (mq >= mapq) & (rid >= 0)
    counts = np.zeros(len(bed["chr"]), np.int64)
    names = np.array(refs)
    _accumulate(names[rid[sel]], pos[sel] + 1, bed, counts)     # +1: BAM pos is 0-based here, SAM POS is 1-based
    return counts


def count_bam(bam, bed, mapq):
    counts = np.zeros(len(bed["chr"]), np.int64)
    if shutil.which("samtools") is None:
        print("  samtools not found - using the slow built-in BAM reader", file=sys.stderr)
        return _count_pure_python(bam, bed, mapq)
    cmd = "samtools view -F 0x904 -q %d '%s' 2>/dev/null | cut -f3,4" % (mapq, bam)
    pr = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE)
    rest = b""
    while True:
        data = pr.stdout.read(1 << 25)
        if not data:
            break
        text = rest + data
        cut = text.rfind(b"\n")
        body, rest = text[:cut], text[cut + 1:]
        tok = np.array(body.split())
        if len(tok) >= 2:
            _accumulate(tok[0::2], tok[1::2].astype(np.int64), bed, counts)
    if pr.wait() != 0:
        sys.exit("samtools failed on " + bam)
    return counts


# ----------------------------------------------------------------------------- model
def gc_curve(ratio, gc):
    ok = np.isfinite(ratio) & np.isfinite(gc) & (ratio > 0)
    edges = np.linspace(0.25, 0.70, 91)
    idx = np.digitize(gc[ok], edges)
    cen, med = [], []
    for j in np.unique(idx):
        sel = idx == j
        if 1 <= j < len(edges) and sel.sum() >= 25:
            cen.append((edges[j - 1] + edges[j]) / 2)
            med.append(np.median(ratio[ok][sel]))
    if len(cen) < 5:
        return lambda g: np.ones_like(np.asarray(g, float))
    cen, med = np.array(cen), np.array(med)
    k = 5
    sm = np.convolve(np.pad(med, (k // 2, k // 2), mode="edge"), np.ones(k) / k, mode="valid")
    norm = np.median(ratio[ok])
    return lambda g: np.clip(np.interp(np.nan_to_num(np.asarray(g, float), nan=0.41), cen, sm) / norm, 0.2, 5.0)


def build_model(C, bed, a):
    chrom = bed["chr"]
    is_auto = np.isin(chrom, AUTO)
    is_x = chrom == "X"
    static = (bed["nr"] <= a.max_nratio) & np.isfinite(bed["gc"]) & (is_auto | is_x)
    base = static & is_auto
    scale = np.array([np.median(c[base][c[base] > 0]) for c in C])
    norm = C / scale[:, None]
    med = np.median(norm, 0)
    cv = norm.std(0, ddof=1) / np.maximum(norm.mean(0), 1e-9)
    ref_level = np.median(med[base])
    cand = static & (med >= a.cov_min * ref_level) & (med <= a.cov_max * ref_level)
    cvc = cv[cand]
    cut = np.median(cvc) + a.cv_sigma * 1.4826 * np.median(np.abs(cvc - np.median(cvc)))
    keep = cand & (cv <= cut)
    return {"keep": keep, "exp": med, "is_auto": is_auto, "is_x": is_x, "chrom": chrom, "gc": bed["gc"],
            "n_static": int(static.sum()), "n_cand": int(cand.sum()), "n_keep": int(keep.sum()), "cv_cut": float(cut)}


def score(c, m):
    k, ka = m["keep"], m["keep"] & m["is_auto"]
    c = c.astype(float)
    s = c[ka].sum() / m["exp"][ka].sum()
    f = gc_curve(c[ka] / (m["exp"][ka] * s), m["gc"][ka])
    corr = np.where(np.isfinite(m["gc"]), c / f(m["gc"]), c)
    tot = corr[ka].sum()
    frac = {ch: corr[k & (m["chrom"] == ch)].sum() / tot for ch in AUTO + ["X"]}
    r_raw = c[ka] / (m["exp"][ka] * s)
    r_cor = corr[ka] / (m["exp"][ka] * (tot / m["exp"][ka].sum()))
    return frac, {"cv_raw": float(r_raw.std() / r_raw.mean()), "cv_corr": float(r_cor.std() / r_cor.mean()),
                  "poisson_cv": float(1 / np.sqrt(c[ka].mean()))}


def stats(F, groups=None):
    mu, sd = F.mean(0), F.std(0, ddof=1)
    sdg = None
    if groups is not None and len(set(groups)) >= 3:
        G = np.vstack([F[[i for i, g in enumerate(groups) if g == u]].mean(0) for u in dict.fromkeys(groups)])
        sdg = G.std(0, ddof=1)
    return mu, sd, sdg


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--bed", required=True)
    ap.add_argument("--samples", required=True)
    ap.add_argument("--outdir", default="z_out")
    ap.add_argument("--mapq", type=int, default=30)
    ap.add_argument("--max-nratio", type=float, default=0.1)
    ap.add_argument("--cov-min", type=float, default=0.6)
    ap.add_argument("--cov-max", type=float, default=1.6)
    ap.add_argument("--cv-sigma", type=float, default=3.5)
    ap.add_argument("--z-high", type=float, default=3.0)
    ap.add_argument("--exclude-ref", default="", help="comma-separated sample_ids to drop from the reference")
    ap.add_argument("--spike", default="", help="in-silico mixing, e.g. 0.05,0.10,0.15 (needs expected=T21/T18/T13 test samples)")
    a = ap.parse_args()
    os.makedirs(a.outdir + "/counts", exist_ok=True)
    bed = load_bed(a.bed)
    rows = [r for r in csv.DictReader([l for l in open(a.samples) if l.strip() and not l.startswith("#")], delimiter="\t")]
    drop = set(x for x in a.exclude_ref.split(",") if x)
    for r in rows:
        r["role"] = r["role"].strip().lower()
        r["group"] = (r.get("group") or r["sample_id"]).strip()
        r["expected"] = (r.get("expected") or "").strip().upper()
    # counts
    C = {}
    for r in rows:
        sid, f = r["sample_id"], f"{a.outdir}/counts/{r['sample_id']}.npy"
        if os.path.exists(f):
            C[sid] = np.load(f)
        else:
            print("counting", sid, flush=True)
            C[sid] = count_bam(r["bam"], bed, a.mapq)
            np.save(f, C[sid])
        print("  %-8s reads in BED windows: %d" % (sid, C[sid].sum()))
    ref = [r for r in rows if r["role"] == "ref" and r["sample_id"] not in drop]
    test = [r for r in rows if r["role"] == "test"]
    if len(ref) < 4:
        sys.exit("need at least 4 reference samples")
    CR = np.vstack([C[r["sample_id"]] for r in ref]).astype(float)
    m = build_model(CR, bed, a)
    print("\nwindows: BED %d | N-ratio/GC ok (auto+X) %d | after coverage filter %d | kept (stable) %d (%.1f%% of auto+X)" % (
        len(bed["chr"]), m["n_static"], m["n_cand"], m["n_keep"], 100 * m["n_keep"] / max(m["n_static"], 1)))
    print("kept autosomal windows per target chromosome:", {c: int((m["keep"] & (m["chrom"] == c)).sum()) for c in TARGET})
    chs = AUTO + ["X"]
    FR, QR = [], []
    for r in ref:
        f, q = score(C[r["sample_id"]], m)
        FR.append([f[c] for c in chs])
        QR.append(q)
    FR = np.array(FR)
    groups = [r["group"] for r in ref]
    mu, sd, sdg = stats(FR, groups)
    print("independent groups in the reference: %d of %d samples %s" % (len(set(groups)), len(ref), "(SD from replicates is too optimistic)" if len(set(groups)) < len(ref) else ""))
    # leave-one-out
    loo = []
    for i, r in enumerate(ref):
        o = np.delete(FR, i, 0)
        # when replicates exist, remove the whole group of the left-out sample
        keep_idx = [j for j in range(len(ref)) if groups[j] != groups[i]]
        o = FR[keep_idx] if len(keep_idx) >= 4 else o
        loo.append((FR[i] - o.mean(0)) / o.std(0, ddof=1))
    loo = np.array(loo)
    # test samples
    res = []
    for r in test:
        f, q = score(C[r["sample_id"]], m)
        v = np.array([f[c] for c in chs])
        res.append((r, v, (v - mu) / sd, None if sdg is None else (v - mu) / sdg, q))
    # ----- files
    with open(a.outdir + "/reference_stats.tsv", "w") as fh:
        fh.write("chr\tmean_fraction\tSD_all_samples\tCV_%\tSD_between_groups\n")
        for j, c in enumerate(chs):
            fh.write("%s\t%.6f\t%.6f\t%.3f\t%s\n" % (c, mu[j], sd[j], 100 * sd[j] / mu[j], "NA" if sdg is None else "%.6f" % sdg[j]))
    with open(a.outdir + "/reference_loo_z.tsv", "w") as fh:
        fh.write("sample_id\tgroup\t" + "\t".join("chr" + c for c in chs) + "\tmax_abs_Z_autosome\n")
        for i, r in enumerate(ref):
            fh.write("%s\t%s\t%s\t%.2f\n" % (r["sample_id"], r["group"], "\t".join("%.2f" % x for x in loo[i]), np.abs(loo[i][:22]).max()))
    with open(a.outdir + "/z_scores.tsv", "w") as fh:
        fh.write("sample_id\tchr\tfraction\tratio_to_ref_mean\tZ_sd_all\tZ_sd_groups\n")
        for r, v, z, zg, q in res:
            for j, c in enumerate(chs):
                fh.write("%s\t%s\t%.6f\t%.4f\t%.2f\t%s\n" % (r["sample_id"], c, v[j], v[j] / mu[j], z[j], "NA" if zg is None else "%.2f" % zg[j]))
    # ----- report on screen
    print("\n=== REFERENCE: chromosome fraction CV (SD_all / mean) for target chromosomes ===")
    for c in TARGET:
        j = chs.index(c)
        print("chr%-3s mean %.5f  SD %.6f  CV %.2f%%   (SD between groups: %s)" % (c, mu[j], sd[j], 100 * sd[j] / mu[j], "NA" if sdg is None else "%.6f" % sdg[j]))
    print("\n=== REFERENCE leave-one-out Z (own group removed) : sample, Z21, Z18, Z13, max|Z| over autosomes ===")
    for i, r in enumerate(ref):
        flag = "  <-- check" if np.abs(loo[i][:22]).max() >= a.z_high else ""
        print("%-8s %6.2f %6.2f %6.2f   %5.2f (chr%s)%s" % (r["sample_id"], loo[i][chs.index("21")], loo[i][chs.index("18")], loo[i][chs.index("13")],
                                                       np.abs(loo[i][:22]).max(), chs[int(np.abs(loo[i][:22]).argmax())], flag))
    print("\n=== TEST samples ===")
    print("%-8s %-9s %9s %9s %9s %9s | worst other autosome | CV raw->GC-corr (Poisson)" % ("sample", "expected", "Z21", "Z18", "Z13", "Z chrX"))
    verdicts = []
    for r, v, z, zg, q in res:
        zt = {c: z[chs.index(c)] for c in TARGET}
        oth = [(abs(z[j]), chs[j], z[j]) for j in range(22) if chs[j] not in TARGET]
        w = max(oth)
        exp = r["expected"]
        if exp.startswith("T"):
            ok = zt[exp[1:]] >= a.z_high
            verdicts.append((r["sample_id"], exp, "PASS" if ok else "FAIL"))
        elif exp == "EUPLOID":
            ok = all(abs(zt[c]) < a.z_high for c in TARGET)
            verdicts.append((r["sample_id"], exp, "PASS" if ok else "FAIL"))
        print("%-8s %-9s %9.1f %9.1f %9.1f %9.1f | chr%-3s Z=%7.1f        | %.1f%% -> %.1f%% (%.1f%%)" % (
            r["sample_id"], exp or "-", zt["21"], zt["18"], zt["13"], z[chs.index("X")], w[1], w[2],
            100 * q["cv_raw"], 100 * q["cv_corr"], 100 * q["poisson_cv"]))
        if zg is not None:
            print("%-8s %-9s %9.1f %9.1f %9.1f %9s |  (same, SD between groups)" % ("", "", zg[chs.index("21")], zg[chs.index("18")], zg[chs.index("13")], ""))
        print("%-8s %-9s ratio to ref mean: 21 %.3f  18 %.3f  13 %.3f  (trisomy in 100%% of cells -> ~1.5 x share of the chromosome)" % (
            "", "", v[chs.index("21")] / mu[chs.index("21")], v[chs.index("18")] / mu[chs.index("18")], v[chs.index("13")] / mu[chs.index("13")]))
    if any(v[1].startswith("T") for v in verdicts):
        print("NOTE: negative Z on the OTHER chromosomes is expected for a sample whose trisomy is in 100% of cells (gDNA): the extra "
              "chromosome raises the total, so every other fraction falls by ~(share x 0.5). At a real fetal fraction of 10% this shift is ~10x smaller.")
    for s, e, vd in verdicts:
        print("verdict %-8s expected %-8s -> %s" % (s, e, vd))
    # ----- in-silico mixing (sensitivity)
    if a.spike:
        fr = [float(x) for x in a.spike.split(",")]
        rng = np.random.default_rng(1)
        ka = m["keep"] & m["is_auto"]
        print("\n=== IN-SILICO MIXING: reads of the aneuploid sample mixed into each reference sample (fetal fraction = mixing fraction) ===")
        print("Z of the expected chromosome: mean [min .. max] over base reference samples; leave-one-group-out statistics")
        for r, v, z, zg, q in res:
            if not r["expected"].startswith("T"):
                continue
            tc = r["expected"][1:]
            ct = C[r["sample_id"]].astype(float)
            for fmix in fr:
                zs = []
                for i, b in enumerate(ref):
                    cb = C[b["sample_id"]].astype(float)
                    mix = (1 - fmix) * cb / cb[ka].sum() + fmix * ct / ct[ka].sum()
                    cnt = rng.poisson(mix * cb[ka].sum())
                    f, _ = score(cnt, m)
                    kk = [j for j in range(len(ref)) if groups[j] != groups[i]]
                    o = FR[kk]
                    zs.append((f[tc] - o.mean(0)[chs.index(tc)]) / o.std(0, ddof=1)[chs.index(tc)])
                print("%-8s chr%-3s FF %4.0f%%: Z = %6.1f [%6.1f .. %6.1f]" % (r["sample_id"], tc, 100 * fmix, np.mean(zs), np.min(zs), np.max(zs)))
    # ----- figure
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [2.4, 1]})
        w = 0.8 / max(len(res), 1)
        for t, (r, v, z, zg, q) in enumerate(res):
            ax[0].bar(np.arange(len(chs)) + t * w, z, w, label=r["sample_id"])
        ax[0].set_yscale("symlog", linthresh=3)
        for y in (-a.z_high, a.z_high):
            ax[0].axhline(y, c="k", ls=":", lw=.8)
        ax[0].set_xticks(np.arange(len(chs)) + 0.4)
        ax[0].set_xticklabels(chs, fontsize=8)
        ax[0].set_ylabel("Z (symlog)")
        ax[0].set_title("Test samples: Z by chromosome")
        ax[0].legend(fontsize=8)
        for i, r in enumerate(ref):
            ax[1].scatter([0, 1, 2], [loo[i][chs.index(c)] for c in TARGET], s=18, alpha=.8)
        ax[1].set_xticks([0, 1, 2])
        ax[1].set_xticklabels(["chr21", "chr18", "chr13"])
        ax[1].axhline(0, c="k", lw=.5)
        for y in (-a.z_high, a.z_high):
            ax[1].axhline(y, c="k", ls=":", lw=.8)
        ax[1].set_title("Reference: leave-one-out Z")
        fig.tight_layout()
        fig.savefig(a.outdir + "/z_overview.png", dpi=130)
    except ImportError:
        pass
    print("\nfiles in %s: z_scores.tsv reference_stats.tsv reference_loo_z.tsv z_overview.png counts/" % a.outdir)


if __name__ == "__main__":
    main()


In [ ]:
python3 nipt_zscore.py --bed hs37d5.100kb.gc.noPAR.new.bed \
  --samples samples.tsv --outdir z_out --spike 0.05,0.10,0.15